# Compare, freeze, evaluate, and demonstrate a calibrated ModernBERT router

This notebook runs one immutable slot from a six-run evidence plan. Change only
`RUN_ID` to produce random seeds 42/43/44 or separate dataset-OOD seeds 42/43/44.
Every slot uses the same replacement safety objective, three training setups,
gates, candidate panel, and
analytical candidate-latency assumptions.

The notebook answers four questions without mixing their evidence:

1. **Which training setup looks safest and most useful on validation?**
2. **Does that one frozen setup pass the sealed test?**
3. **How long does ModernBERT itself take on this Colab GPU?**
4. **What decision would the exported router show in an interactive demo?**

Only ModernBERT tokenization and inference are measured. Fin-R1 and Qwen3-8B
are never loaded or timed; their latency remains analytical.

A numerical example of the stability rule: if only threshold `0.910` passes,
the router stays off. At least two neighboring grid values must pass. A separate
timing example: a measured 12 ms ModernBERT call is compared with the contract's
4 ms nominal, 20 ms conservative, and policy-specific break-even overheads, but
does not retroactively change the frozen policy.

> A passing run means one run passed. Investment-grade evidence still requires
> the complete multi-seed table, dataset-OOD results, and target-hardware timing.


## 1. Set up Colab

The cell installs the current `develop` branch once and forces imports to
come from its `src` directory. The exported manifest records the Git commit,
package versions, Python version, GPU, CUDA runtime, and benchmark fingerprint.

In [ ]:
%cd /content
!test -d /content/LLM_Router || git clone --branch develop https://github.com/BrunoVitti96/LLM-router.git /content/LLM_Router
!git -C /content/LLM_Router pull --ff-only origin develop
%cd /content/LLM_Router
%pip install -q -U ".[notebook]"

import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/LLM_Router")
SOURCE_ROOT = str(PROJECT_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
for module_name in list(sys.modules):
    if module_name == "llm_router" or module_name.startswith("llm_router."):
        del sys.modules[module_name]
importlib.invalidate_caches()
llm_router = importlib.import_module("llm_router")

print(f"Router package ready from {Path(llm_router.__file__).resolve()}")

## 2. Select one frozen experiment slot

`RUN_ID` is the only study-control value to change between Colab sessions. The
run plan contains three random-split feasibility runs and three dataset-OOD
stress tests. Each produces a separate ZIP so a Colab timeout cannot erase the
whole study and a later result cannot alter an earlier run.

The training menu is imported from `llm_router.experiment_plan` rather than
rewritten in the notebook:

- `hybrid_r4`: rank-4 hybrid objective;
- `safety_only_r4`: rank-4 safety-only ablation; and
- `hybrid_r4_dataset_balanced`: equal-dataset sampling ablation.

For example, `RUN_ID = "random_seed_43"` deterministically implies random split
and seed 43. `RUN_ID = "dataset_ood_seed_43"` uses the same seed and training
contract but holds out entire datasets.


In [ ]:
import shutil
import tarfile
from dataclasses import replace
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display

from llm_router.config import DEFAULT_CONFIG
from llm_router.experiment_comparison import (
    build_setup_comparison,
    choose_validation_setup,
    combine_threshold_searches,
)
from llm_router.experiment_plan import (
    ANALYTICAL_SCENARIO_AS_OF,
    FROZEN_EXPERIMENT_PLAN,
    frozen_setup_dicts,
    get_experiment_run,
)
from llm_router.hybrid_inference import (
    HybridModernBERTRouterRuntime,
    create_gradio_demo,
)
from llm_router.modernbert_poc import (
    export_modernbert_hybrid_poc,
    train_modernbert_hybrid_poc,
)
from llm_router.oracle import oracle_choices
from llm_router.public_benchmark import (
    EconomicsScenario,
    ModelProfile,
    benchmark_inventory,
    export_public_benchmark,
    load_llmrouterbench,
    make_complete_panel,
    run_public_benchmark,
    select_validation_policy,
    simulate_economics,
    split_benchmark,
)
from llm_router.router_overhead import benchmark_modernbert_overhead
from llm_router.utils.training import seed_everything

RUN_ID = "random_seed_43"
RUN_SPEC = get_experiment_run(RUN_ID)
SEED = RUN_SPEC.seed
SPLIT_MODE = RUN_SPEC.split_mode
EPOCHS = 8
MINIMUM_EPOCHS = 2
EARLY_STOPPING_PATIENCE = 2
MINIMUM_CONSECUTIVE_FEASIBLE_THRESHOLDS = (
    DEFAULT_CONFIG.minimum_consecutive_feasible_thresholds
)
DATA_ROOT = Path("/content/LLMRouterBench")
OUTPUT_DIR = Path(f"reports_benchmark/{RUN_ID}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MEASURE_ROUTER_OVERHEAD = True
ROUTER_OVERHEAD_TIMED_REQUESTS = 100
ROUTER_OVERHEAD_WARMUP_REQUESTS = 10
LAUNCH_INTERACTIVE_DEMO = True
SETUP_SPECS = frozen_setup_dicts()

def log_stage(stage, **values):
    timestamp = datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
    details = " | ".join(f"{key}={value}" for key, value in values.items())
    print(f"[{timestamp}] {stage}" + (f" | {details}" if details else ""))

seed_everything(SEED)
display(
    pd.DataFrame(
        [
            {
                "run_id": run.run_id,
                "split_mode": run.split_mode,
                "seed": run.seed,
                "purpose": run.purpose,
                "selected_now": run.run_id == RUN_ID,
            }
            for run in FROZEN_EXPERIMENT_PLAN
        ]
    )
)
log_stage(
    "experiment configured",
    run_id=RUN_ID,
    device=DEVICE,
    split=SPLIT_MODE,
    seed=SEED,
    setups=len(SETUP_SPECS),
    stable_thresholds=MINIMUM_CONSECUTIVE_FEASIBLE_THRESHOLDS,
)
if DEVICE == "cpu":
    print("Warning: use a Colab GPU; the three-setup comparison is slow on CPU.")


## 3. Download and inventory the benchmark

The benchmark already contains candidate answers and quality scores. Candidate
LLMs are never loaded during router training. The archive download is public;
an absent Hugging Face token is only a warning.

In [ ]:
inventory = benchmark_inventory(DATA_ROOT)
archive_preview = []
if inventory.empty:
    archive = hf_hub_download(
        repo_id="NPULH/LLMRouterBench",
        filename="bench-release.tar.gz",
        repo_type="dataset",
    )
    results_dir = DATA_ROOT / "results"
    results_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as bundle:
        archive_preview = bundle.getnames()[:12]
        bundle.extractall(results_dir, filter="data")
    inventory = benchmark_inventory(DATA_ROOT)
if inventory.empty:
    extracted_preview = [
        str(path.relative_to(DATA_ROOT))
        for path in DATA_ROOT.rglob("*")
        if path.is_file()
    ][:20]
    raise RuntimeError(
        "No dataset/split/model/*.json layout was found. "
        f"Archive entries: {archive_preview}; files: {extracted_preview}"
    )
DATA_ROOT = Path(inventory.iloc[0]["file"]).parents[3]
inventory_summary = (
    inventory.groupby(["model", "dataset", "source_split"])
    .size()
    .rename("files")
    .reset_index()
)
log_stage(
    "benchmark ready",
    models=inventory.model.nunique(),
    datasets=inventory.dataset.nunique(),
    files=len(inventory),
    root=DATA_ROOT,
)
display(inventory_summary)

## 4. Declare a non-dominated candidate panel

Fin-R1 is the faster 7B replacement and Qwen3-8B is the 8.2B fallback
candidate. Both are treated as dense autoregressive BF16 models. A future
candidate should be added only with pre-collected quality and sourced model
facts, then screened for nonzero oracle selection and meaningful speedup.

In [ ]:
MODEL_PROFILES = {
    "Fin-R1": {
        "parameters_billions": 7.0,
        "active_parameters_billions": 7.0,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
    "Qwen3-8B": {
        "parameters_billions": 8.2,
        "active_parameters_billions": 8.2,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
}
SELECTED_MODELS = tuple(MODEL_PROFILES)
missing_models = sorted(set(SELECTED_MODELS) - set(inventory.model))
assert not missing_models, f"Missing configured models: {missing_models}"
print("Candidate panel:", SELECTED_MODELS)

## 5. Build analytical latency and audit leakage

Expected output length uses prompt size only:

$$\widehat n_{out}=\operatorname{clip}(24+0.2n,16,256).$$

The counterfactual check multiplies recorded completion lengths by 100 and
requires analytical latency to remain identical. This proves realized answer
length cannot leak into routing labels.

In [ ]:
profiles = tuple(
    ModelProfile(
        name=name,
        input_price_per_million=0.0,
        output_price_per_million=0.0,
        **settings,
    )
    for name, settings in MODEL_PROFILES.items()
)
scenario = EconomicsScenario(
    name="notebook-analytical-latency-poc",
    as_of=ANALYTICAL_SCENARIO_AS_OF,
    profiles=profiles,
    latency_method="analytical",
    effective_tflops=60.0,
    memory_bandwidth_gbps=900.0,
    fixed_model_overhead_s=0.015,
    router_overhead_s=0.004,
    output_base_tokens=24.0,
    output_tokens_per_prompt_token=0.20,
    output_min_tokens=16,
    output_max_tokens=256,
    notes="POC assumptions; not measured production latency.",
)

records = load_llmrouterbench(DATA_ROOT, models=SELECTED_MODELS)
simulated = simulate_economics(records, scenario)
panel = make_complete_panel(simulated, models=SELECTED_MODELS)
assert simulated.latency_source.eq("analytical").all()

counterfactual_records = records.copy()
counterfactual_records["completion_tokens"] *= 100
counterfactual = simulate_economics(counterfactual_records, scenario)
assert np.allclose(
    simulated.simulated_latency_s,
    counterfactual.simulated_latency_s,
)
latency_summary = simulated.groupby("model")["simulated_latency_s"].agg(
    ["min", "median", "mean", "max"]
)
log_stage(
    "Leakage check passed",
    complete_prompts=len(panel.examples),
    repeated_prompt_rows=int((panel.examples.prompt_group_size > 1).sum()),
    largest_prompt_group=int(panel.examples.prompt_group_size.max()),
)
display(latency_summary)

## 6. Freeze train, validation, and sealed-test splits

The fallback is selected from training quality only. Exact normalized prompt
hashes are group-disjoint, so the same question cannot cross splits under a
different benchmark ID. Validation may select checkpoints, calibration,
setup, threshold, and activation. Test outcomes remain sealed.

In [ ]:
split = split_benchmark(panel, mode=SPLIT_MODE, seed=SEED)
split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "prompts": [len(split.train), len(split.validation), len(split.test)],
        "datasets": [
            len(split.train_datasets),
            len(split.validation_datasets),
            len(split.test_datasets),
        ],
    }
)
split_hashes = {
    name: set(panel.examples.iloc[indices].prompt_hash)
    for name, indices in {
        "train": split.train,
        "validation": split.validation,
        "test": split.test,
    }.items()
}
assert split_hashes["train"].isdisjoint(split_hashes["validation"])
assert split_hashes["train"].isdisjoint(split_hashes["test"])
assert split_hashes["validation"].isdisjoint(split_hashes["test"])
if SPLIT_MODE == "dataset_ood":
    assert set(split.train_datasets).isdisjoint(split.validation_datasets)
    assert set(split.train_datasets).isdisjoint(split.test_datasets)
    assert set(split.validation_datasets).isdisjoint(split.test_datasets)
log_stage("split frozen", **dict(zip(split_summary["split"], split_summary["prompts"])))
display(split_summary)

## 7. Verify validation-only oracle headroom and scenario sensitivity

The hindsight oracle sees recorded outcomes and chooses the fastest safe model.
It is not deployable; it measures the opportunity ceiling. If oracle savings
are near zero, changing ModernBERT cannot rescue the candidate panel.

In [ ]:
def validation_oracle_metrics(candidate_panel):
    fallback_index = int(candidate_panel.score[split.train].mean(axis=0).argmax())
    targets = oracle_choices(
        candidate_panel.score,
        candidate_panel.latency,
        fallback_index=fallback_index,
    )
    indices = split.validation
    rows = np.arange(len(indices))
    chosen = targets[indices]
    fallback_quality = candidate_panel.score[indices, fallback_index].mean()
    oracle_quality = candidate_panel.score[indices][rows, chosen].mean()
    fallback_latency = candidate_panel.latency[indices, fallback_index].mean()
    oracle_latency = candidate_panel.latency[indices][rows, chosen].mean()
    return {
        "fallback_model": candidate_panel.models[fallback_index],
        "quality_retention": oracle_quality / max(fallback_quality, 1e-12),
        "latency_savings": 1 - oracle_latency / fallback_latency,
        "fallback_usage": np.mean(chosen == fallback_index),
    }

sensitivity_settings = {
    "balanced": {},
    "compute_conservative": {"effective_tflops": 40.0},
    "bandwidth_conservative": {"memory_bandwidth_gbps": 600.0},
    "larger_fixed_overhead": {"fixed_model_overhead_s": 0.050},
    "longer_outputs": {
        "output_base_tokens": 48.0,
        "output_tokens_per_prompt_token": 0.35,
    },
}
sensitivity_rows = []
for name, overrides in sensitivity_settings.items():
    variant = replace(scenario, name=name, **overrides)
    variant_records = simulate_economics(records, variant)
    variant_panel = make_complete_panel(variant_records, models=SELECTED_MODELS)
    sensitivity_rows.append(
        {"scenario": name, **validation_oracle_metrics(variant_panel)}
    )
sensitivity = pd.DataFrame(sensitivity_rows)
assert (sensitivity.latency_savings > 0).all()
log_stage(
    "oracle headroom confirmed",
    minimum_savings=f"{sensitivity.latency_savings.min():.2%}",
    maximum_savings=f"{sensitivity.latency_savings.max():.2%}",
)
display(sensitivity)

## 8. Understand the objective, loss, and calibration

Replacement safety is

$$y_m(x)=\mathbf 1[Q_m(x)\ge Q_f(x)-\epsilon_q].$$

The deployable head uses class-balanced BCE. The optional training-only head
imitates the hindsight oracle and penalizes quality risk plus latency regret:

$$\mathcal L=\mathcal L_{balanced\ safety}
+\lambda_{oracle}\mathcal L_{oracle}.$$

The comparison then asks what changes in practice. Comparing `safety_only_r4` ($\lambda_{oracle}=0$) against `hybrid_r4`
($\lambda_{oracle}=0.25$) directly measures whether the auxiliary loss helps.
Comparing ordinary and dataset-balanced sampling measures whether large datasets
are dominating the learned policy. Optional rank 8 measures adapter capacity.

Every safety logit is Platt-calibrated. Validation threshold selection uses
out-of-fold probabilities, so no validation row calibrates itself.

## 9. Train every declared setup with epoch-level logs

The logs show training and validation loss, current best epoch, cumulative
skipped mixed-precision steps, and early stopping. A falling training loss
with rising validation loss indicates overfitting; simply adding epochs is
then unlikely to help.

In [ ]:
def make_epoch_logger(setup_name):
    def report(row):
        marker = "BEST" if row["is_best_epoch"] else "    "
        stop = " | early-stop" if row["will_stop_early"] else ""
        print(
            f"[{setup_name}] epoch={int(row['epoch'])}/{EPOCHS} {marker} "
            f"train={row['train_total_loss']:.4f} "
            f"validation={row['validation_total_loss']:.4f} "
            f"best_epoch={int(row['best_epoch_so_far'])} "
            f"seconds={row['epoch_seconds']:.1f} "
            f"train_examples_per_second={row['train_examples_per_second']:.1f} "
            f"skipped_steps_total={int(row['skipped_optimizer_steps'])}"
            f"{stop}"
        )
    return report

trainings = {}
setup_configs = {}
for setup_name, spec in SETUP_SPECS.items():
    log_stage("training started", setup=setup_name, **spec)
    setup_config = replace(
        DEFAULT_CONFIG,
        seed=SEED,
        lora_r=spec["lora_r"],
        lora_alpha=spec["lora_alpha"],
    )
    setup_configs[setup_name] = setup_config
    training = train_modernbert_hybrid_poc(
        panel,
        split,
        config=setup_config,
        epochs=EPOCHS,
        batch_size=8,
        learning_rate=1e-4,
        head_learning_rate=2e-4,
        minimum_epochs=MINIMUM_EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        quality_epsilon=0.0,
        safety_loss_weight=1.0,
        oracle_auxiliary_weight=spec["oracle_auxiliary_weight"],
        dataset_balanced_sampling=spec["dataset_balanced_sampling"],
        device=DEVICE,
        progress_callback=make_epoch_logger(setup_name),
    )
    training.model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    trainings[setup_name] = training
    log_stage(
        "training finished",
        setup=setup_name,
        best_epoch=training.best_epoch,
        epochs=training.epochs_completed,
        minutes=f"{training.training_seconds / 60:.1f}",
        truncation=f"{training.input_diagnostics['truncation_rate']:.2%}",
    )

## 10. Compare setups on validation only

Every threshold now reports each individual gate, total gates passed, and
contiguous feasible-block size. The setup leaderboard prioritizes:

1. an active policy that passes all gates at neighboring thresholds;
2. conservative-overhead savings;
3. routed-safety precision LCB; and
4. validation loss as the final tie-breaker.

No sealed-test score is used here.

In [ ]:
policy_kwargs = dict(  # noqa: C408
    objective="latency",
    minimum_quality_retention=DEFAULT_CONFIG.minimum_quality_retention,
    confidence=DEFAULT_CONFIG.quality_confidence,
    validation_quality_margin=DEFAULT_CONFIG.validation_quality_margin,
    minimum_predicted_savings=DEFAULT_CONFIG.minimum_predicted_speedup,
    router_overhead_s=scenario.router_overhead_s,
    conservative_router_overhead_s=(
        DEFAULT_CONFIG.conservative_router_overhead_s
    ),
    minimum_macro_quality_retention=(
        DEFAULT_CONFIG.minimum_macro_quality_retention
    ),
    maximum_quality_loss_rate_ucl=(
        DEFAULT_CONFIG.maximum_quality_loss_rate_ucl
    ),
    minimum_routed_safety_precision_lcb=(
        DEFAULT_CONFIG.minimum_routed_safety_precision_lcb
    ),
    minimum_guarded_dataset_quality_retention_lcb=(
        DEFAULT_CONFIG.minimum_guarded_dataset_quality_retention_lcb
    ),
    minimum_guarded_dataset_prompts=(
        DEFAULT_CONFIG.minimum_guarded_dataset_prompts
    ),
    minimum_consecutive_feasible_thresholds=(
        MINIMUM_CONSECUTIVE_FEASIBLE_THRESHOLDS
    ),
    seed=SEED,
)

selections = {}
for setup_name, training in trainings.items():
    selections[setup_name] = select_validation_policy(
        panel,
        split,
        routing_probabilities=training.safety_probabilities,
        router_name=f"modernbert_hybrid_router__{setup_name}",
        **policy_kwargs,
    )
    selection = selections[setup_name]
    log_stage(
        "validation policy evaluated",
        setup=setup_name,
        active=selection.router_active,
        selected_threshold=selection.selected_threshold,
        diagnostic_threshold=selection.diagnostic_threshold,
        reason="; ".join(selection.failure_reasons) or "all gates passed",
    )

setup_comparison = build_setup_comparison(trainings, selections)
setup_threshold_search = combine_threshold_searches(selections)
display(
    setup_comparison.sort_values(
        ["router_active", "conservative_resource_savings"],
        ascending=False,
    ).style.format(
        {
            "quality_retention_lcb": "{:.2%}",
            "routed_safety_precision_lcb": "{:.2%}",
            "safe_opportunity_recall": "{:.2%}",
            "routed_fraction": "{:.2%}",
            "nominal_resource_savings": "{:.2%}",
            "conservative_resource_savings": "{:.2%}",
            "calibrated_ece": "{:.3f}",
        }
    )
)

## 11. Visualize what helps and what blocks activation

- Training curves expose overfitting.
- The scatter plot shows the safety/latency tradeoff under 20 ms overhead.
- Threshold frontiers show whether a setup has a broad safe region or only
  an isolated lucky threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for setup_name, training in trainings.items():
    history = training.history
    axes[0].plot(history.epoch, history.train_total_loss, "--", label=f"{setup_name} train")
    axes[0].plot(history.epoch, history.validation_total_loss, label=f"{setup_name} validation")
axes[0].set(title="Training versus validation loss", xlabel="Epoch", ylabel="Total loss")
axes[0].legend(fontsize=8, ncol=2)
axes[0].grid(alpha=0.25)

colors = setup_comparison.router_active.map({True: "tab:green", False: "tab:red"})
axes[1].scatter(
    100 * setup_comparison.conservative_resource_savings,
    100 * setup_comparison.routed_safety_precision_lcb,
    s=100 + 700 * setup_comparison.routed_fraction,
    c=colors,
    alpha=0.8,
)
for row in setup_comparison.itertuples():
    axes[1].annotate(
        row.setup,
        (100 * row.conservative_resource_savings, 100 * row.routed_safety_precision_lcb),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
    )
axes[1].axhline(100 * DEFAULT_CONFIG.minimum_routed_safety_precision_lcb, color="black", linestyle="--", label="precision gate")
axes[1].axvline(0, color="black", linestyle=":", label="break-even")
axes[1].set(
    title="Validation safety versus 20 ms savings",
    xlabel="Conservative analytical savings (%)",
    ylabel="Routed-safety precision LCB (%)",
)
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharex=True)
for setup_name, frontier in setup_threshold_search.groupby("setup"):
    axes[0].plot(frontier.threshold, 100 * frontier.quality_retention_lcb, marker=".", label=setup_name)
    axes[1].plot(frontier.threshold, 100 * frontier.routed_safety_precision_lcb, marker=".", label=setup_name)
    axes[2].plot(frontier.threshold, 100 * frontier.conservative_resource_savings, marker=".", label=setup_name)
axes[0].axhline(100 * (DEFAULT_CONFIG.minimum_quality_retention + DEFAULT_CONFIG.validation_quality_margin), color="black", linestyle="--")
axes[1].axhline(100 * DEFAULT_CONFIG.minimum_routed_safety_precision_lcb, color="black", linestyle="--")
axes[2].axhline(0, color="black", linestyle="--")
axes[0].set(title="Quality-retention LCB", ylabel="Percent")
axes[1].set(title="Routed-safety precision LCB")
axes[2].set(title="Savings at 20 ms overhead")
for axis in axes:
    axis.set_xlabel("Safety threshold")
    axis.grid(alpha=0.25)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

gate_columns = [column for column in setup_threshold_search if column.startswith("passes_") and column.endswith("_gate")]
gate_failures = (
    setup_threshold_search.groupby("setup")[gate_columns]
    .apply(lambda frame: (~frame).sum())
    .astype(int)
)
print("Number of thresholds blocked by each gate (lower is better):")
display(gate_failures)

calibration_rows = []
for setup_name, training in trainings.items():
    diagnostics = training.calibration_diagnostics
    calibration_rows.append(
        {
            "setup": setup_name,
            "raw_brier": diagnostics.raw_brier.mean(),
            "calibrated_brier": diagnostics.calibrated_brier.mean(),
            "raw_ece": diagnostics.raw_ece.mean(),
            "calibrated_ece": diagnostics.calibrated_ece.mean(),
        }
    )
calibration_plot = pd.DataFrame(calibration_rows).set_index("setup")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
calibration_plot[["raw_brier", "calibrated_brier"]].plot.bar(ax=axes[0])
calibration_plot[["raw_ece", "calibrated_ece"]].plot.bar(ax=axes[1])
axes[0].set(title="Brier score: lower is better", ylabel="Score")
axes[1].set(title="Expected calibration error: lower is better", ylabel="ECE")
for axis in axes:
    axis.tick_params(axis="x", rotation=20)
    axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 12. Freeze one setup before opening the sealed test

Selection is deterministic and validation-only. If every setup fails, the
best diagnostic near-miss is retained but its final policy remains fallback
only. Printing `SELECTED_SETUP` here creates a clear audit boundary.

In [ ]:
SELECTED_SETUP = choose_validation_setup(setup_comparison)
setup_comparison["selected_for_test"] = setup_comparison.setup.eq(SELECTED_SETUP)
selected_training = trainings[SELECTED_SETUP]
selected_config = setup_configs[SELECTED_SETUP]
selected_spec = SETUP_SPECS[SELECTED_SETUP]
frozen_validation_policy = selections[SELECTED_SETUP]

log_stage(
    "POLICY FROZEN — sealed test may now open",
    setup=SELECTED_SETUP,
    active=frozen_validation_policy.router_active,
    threshold=frozen_validation_policy.selected_threshold,
    feasible_block=int(
        setup_comparison.set_index("setup").loc[
            SELECTED_SETUP, "feasible_block_size"
        ]
    ),
)

## 13. Measure ModernBERT overhead without timing candidate LLMs

This diagnostic uses deterministic validation-prompt samples at batch size one.
It measures ModernBERT tokenization, tensor transfer, and router inference. It
does not load Fin-R1 or Qwen3-8B, and it does not change the already-frozen
threshold or the 4 ms/20 ms validation gates.

Both model-only and end-to-end distributions are exported. The end-to-end p50
and p95 are the relevant numbers for comparing with the policy's break-even
router overhead.


In [ ]:
router_overhead_benchmark = None
if MEASURE_ROUTER_OVERHEAD:
    validation_examples = panel.examples.iloc[split.validation]
    router_overhead_benchmark = benchmark_modernbert_overhead(
        selected_training.model,
        selected_training.tokenizer,
        validation_examples.prompt.to_numpy(),
        validation_examples.prompt_tokens.to_numpy(),
        device=DEVICE,
        max_input_tokens=selected_config.max_input_tokens,
        timed_requests=ROUTER_OVERHEAD_TIMED_REQUESTS,
        warmup_requests=ROUTER_OVERHEAD_WARMUP_REQUESTS,
        seed=SEED,
    )
    end_to_end = router_overhead_benchmark.summary["end_to_end_ms"]
    log_stage(
        "ModernBERT overhead measured",
        p50_ms=f"{end_to_end['p50']:.2f}",
        p95_ms=f"{end_to_end['p95']:.2f}",
        candidate_latency="analytical-only",
        policy_changed=False,
    )
    display(pd.DataFrame(router_overhead_benchmark.summary).loc[
        ["mean", "p50", "p95", "maximum"],
        ["end_to_end_ms", "model_only_ms"],
    ])
else:
    print("ModernBERT timing disabled; frozen 4 ms and 20 ms assumptions remain.")


## 14. Open the sealed test exactly once

`run_public_benchmark` repeats the frozen validation selection internally as
a consistency check, then evaluates the selected policy on test. The field
`single_run_passed` is the correct interpretation; multi-seed and OOD evidence
are still outstanding.

In [ ]:
result = run_public_benchmark(
    panel,
    split,
    routing_probabilities=selected_training.safety_probabilities,
    router_name="modernbert_hybrid_router",
    selected_setup=SELECTED_SETUP,
    decision_metadata={
        "router_input_tokens": selected_training.router_input_lengths,
        "router_was_truncated": selected_training.router_was_truncated,
    },
    **policy_kwargs,
)
assert result.selected_threshold == frozen_validation_policy.selected_threshold
assert result.router_active == frozen_validation_policy.router_active

router_metrics = result.summary.loc["modernbert_hybrid_router"]
routed = result.decisions.selected_model.ne(result.fallback_model)
gained = result.decisions.quality_delta.gt(0)
lost = result.decisions.quality_delta.lt(0)
log_stage(
    "sealed test complete",
    single_run_passed=result.single_run_passed,
    routed=f"{routed.mean():.2%}",
    gained=int(gained.sum()),
    lost=int(lost.sum()),
    net_answers=int(gained.sum() - lost.sum()),
    retention_lcb=f"{router_metrics.quality_retention_lcb:.2%}",
    savings_4ms=f"{router_metrics.resource_savings:.2%}",
    savings_20ms=f"{router_metrics.conservative_resource_savings:.2%}",
)
display(result.summary)
display(result.router_overhead_sensitivity)
if result.failure_reasons:
    print("Failure reasons:")
    for reason in result.failure_reasons:
        print("-", reason)

break_even_ms = float(
    result.router_overhead_sensitivity.break_even_router_overhead_ms.iloc[0]
)
overhead_comparison = pd.DataFrame(
    [
        {"source": "frozen nominal assumption", "overhead_ms": 4.0},
        {"source": "frozen conservative assumption", "overhead_ms": 20.0},
    ]
)
if router_overhead_benchmark is not None:
    measured = router_overhead_benchmark.summary["end_to_end_ms"]
    overhead_comparison = pd.concat(
        [
            overhead_comparison,
            pd.DataFrame(
                [
                    {"source": "measured ModernBERT p50", "overhead_ms": measured["p50"]},
                    {"source": "measured ModernBERT p95", "overhead_ms": measured["p95"]},
                ]
            ),
        ],
        ignore_index=True,
    )
overhead_comparison["break_even_ms"] = break_even_ms
overhead_comparison["below_break_even"] = (
    overhead_comparison.overhead_ms < overhead_comparison.break_even_ms
)
display(overhead_comparison)


## 15. Diagnose task-mix sensitivity and truncation

Aggregate quality can hide one dataset paying for another. The first plot
shows net answers gained or lost per dataset. The second compares routed
fraction and analytical savings. Truncation is reported separately because
raising the input limit may improve recall but also increase router overhead.

In [ ]:
decisions = result.decisions.assign(
    gained=result.decisions.quality_delta.gt(0),
    lost=result.decisions.quality_delta.lt(0),
    routed=routed,
)
dataset_outcomes = decisions.groupby("dataset").agg(
    prompts=("dataset", "size"),
    routed=("routed", "sum"),
    gains=("gained", "sum"),
    losses=("lost", "sum"),
    truncated=("router_was_truncated", "sum"),
)
dataset_outcomes["net_answers"] = dataset_outcomes.gains - dataset_outcomes.losses
dataset_outcomes["routed_fraction"] = dataset_outcomes.routed / dataset_outcomes.prompts
router_dataset_metrics = result.per_dataset_metrics.loc[
    result.per_dataset_metrics.strategy.eq("modernbert_hybrid_router")
].set_index("dataset")
dataset_outcomes = dataset_outcomes.join(
    router_dataset_metrics[["resource_savings", "conservative_resource_savings"]]
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ordered = dataset_outcomes.sort_values("net_answers")
axes[0].barh(
    ordered.index,
    ordered.net_answers,
    color=np.where(ordered.net_answers >= 0, "tab:green", "tab:red"),
)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set(title="Net correct answers by dataset", xlabel="Gains minus losses")

savings_order = dataset_outcomes.sort_values("conservative_resource_savings")
axes[1].barh(
    savings_order.index,
    100 * savings_order.conservative_resource_savings,
    color=np.where(savings_order.conservative_resource_savings >= 0, "tab:blue", "tab:orange"),
)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set(title="Savings by dataset at 20 ms overhead", xlabel="Analytical savings (%)")
plt.tight_layout()
plt.show()

truncation_summary = decisions.groupby("router_was_truncated").agg(
    prompts=("dataset", "size"),
    routed=("routed", "sum"),
    gains=("gained", "sum"),
    losses=("lost", "sum"),
    mean_safety_probability=("safety_probability__Fin-R1", "mean"),
)
display(dataset_outcomes.sort_values("net_answers"))
display(truncation_summary)

## 16. Export the complete schema-v5 artifact

In addition to the standard reports, the bundle includes all setup comparison
rows and threshold frontiers. The manifest uses `single_run_passed`; the old
`poc_passed` key remains only as a compatibility alias.

In [ ]:
report_dir = export_public_benchmark(result, scenario, OUTPUT_DIR)
sensitivity.to_csv(report_dir / "validation_sensitivity.csv", index=False)
setup_comparison.to_csv(report_dir / "setup_comparison.csv", index=False)
setup_threshold_search.to_csv(
    report_dir / "setup_threshold_search.csv", index=False
)
for setup_name, training in trainings.items():
    setup_dir = report_dir / "setup_diagnostics" / setup_name
    setup_dir.mkdir(parents=True, exist_ok=True)
    training.history.to_csv(setup_dir / "training_history.csv", index=False)
    training.calibration_diagnostics.to_csv(
        setup_dir / "calibration_diagnostics.csv", index=False
    )

artifact_dir = export_modernbert_hybrid_poc(
    selected_training,
    panel.models,
    report_dir / "modernbert_router",
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    poc_passed=result.single_run_passed,
    failure_reasons=result.failure_reasons,
    minimum_predicted_savings=DEFAULT_CONFIG.minimum_predicted_speedup,
    validation_quality_margin=DEFAULT_CONFIG.validation_quality_margin,
    minimum_macro_quality_retention=(
        DEFAULT_CONFIG.minimum_macro_quality_retention
    ),
    maximum_quality_loss_rate_ucl=(
        DEFAULT_CONFIG.maximum_quality_loss_rate_ucl
    ),
    minimum_routed_safety_precision_lcb=(
        DEFAULT_CONFIG.minimum_routed_safety_precision_lcb
    ),
    minimum_guarded_dataset_quality_retention_lcb=(
        DEFAULT_CONFIG.minimum_guarded_dataset_quality_retention_lcb
    ),
    minimum_guarded_dataset_prompts=(
        DEFAULT_CONFIG.minimum_guarded_dataset_prompts
    ),
    conservative_router_overhead_s=(
        DEFAULT_CONFIG.conservative_router_overhead_s
    ),
    minimum_consecutive_feasible_thresholds=(
        MINIMUM_CONSECUTIVE_FEASIBLE_THRESHOLDS
    ),
    benchmark_fingerprint=result.benchmark_fingerprint,
    setup_name=SELECTED_SETUP,
    oracle_auxiliary_weight=selected_spec["oracle_auxiliary_weight"],
    router_overhead_benchmark=(
        router_overhead_benchmark.summary
        if router_overhead_benchmark is not None
        else None
    ),
    config=selected_config,
)
log_stage("artifacts exported", reports=report_dir.resolve(), router=artifact_dir.resolve())
overhead_comparison.to_csv(
    report_dir / "modernbert_overhead_comparison.csv", index=False
)
if router_overhead_benchmark is not None:
    router_overhead_benchmark.export(report_dir)


## 17. Launch the analytical-latency routing demo

The demo reuses the best in-memory checkpoint. For a prompt and candidate-token
estimate it displays calibrated safety, selected model, fallback use, analytical
candidate latencies, the measured ModernBERT call, and estimated net savings.
The candidate LLMs are not downloaded or executed.


In [ ]:
demo_runtime = HybridModernBERTRouterRuntime.from_training_result(
    selected_training,
    model_names=panel.models,
    fallback_model=result.fallback_model,
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    minimum_predicted_savings=DEFAULT_CONFIG.minimum_predicted_speedup,
    scenario=scenario,
    config=selected_config,
    device=DEVICE,
)
demo = create_gradio_demo(demo_runtime)
if LAUNCH_INTERACTIVE_DEMO:
    demo.launch(share=True, debug=False, prevent_thread_lock=True)
else:
    print("Demo prepared. Set LAUNCH_INTERACTIVE_DEMO=True and rerun this cell.")


## 18. Download before Colab shuts down

The ZIP is the authoritative run record. Keep it unchanged, then rerun with
seeds 43 and 44. Dataset-OOD runs must use separate output names.

In [ ]:
bundle_path = shutil.make_archive(
    str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR
)
print(f"Created {bundle_path}")
try:
    from google.colab import files

    files.download(bundle_path)
except ImportError:
    print("Not running in Colab; download the ZIP from the printed path.")

## 19. Final interpretation checklist

Before presenting a result, confirm:

- the ZIP name contains the frozen `RUN_ID`;
- setup comparison used validation only and the sealed test opened once;
- at least two neighboring thresholds passed, or the router failed closed;
- ModernBERT overhead reports p50 and p95 on the named GPU;
- candidate latency is still explicitly analytical and no candidate LLM ran;
- analytical savings remain positive at both 4 ms and 20 ms overhead;
- measured ModernBERT p50/p95 are compared with policy break-even overhead;
- no single dataset supplies all net quality gains;
- random seeds 42, 43, and 44 are reported together without cherry-picking;
- dataset-OOD artifacts remain separate from random-split artifacts; and
- the commercial claim says funded validation, not guaranteed scale benefits.

A fallback-only result proves that the guard worked. It does not prove that the
router generalized. A measured router that exceeds break-even proves that the
current economics need improvement; it is not a reason to hide the timing.
